# Building Agents That Use Code

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

Alfred is planning a party at the Wayne family mansion and needs your help to ensure everything goes smoothly. To assist him, we'll apply what we've learned about how a multi-step `CodeAgent` operates.


## Let's install the dependencies and login to our HF account to access the Inference API

If you haven't installed `smolagents` yet, you can do so by running the following command:

In [1]:
!pip install smolagents -U

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


Let's also login to the Hugging Face Hub to have access to the Inference API.

In [9]:
from huggingface_hub import notebook_login

notebook_login()

## Selecting a Playlist for the Party Using `smolagents`

An important part of a successful party is the music. Alfred needs some help selecting the playlist. Luckily, `smolagents` has got us covered! We can build an agent capable of searching the web. To give the agent access to this tool, we include it in the tool list when creating the agent.

For the model, we'll rely on `InferenceClientModel`, which provides access to Hugging Face's [Inference API](https://huggingface.co/docs/api-inference/index). The default model is `"Qwen/Qwen2.5-Coder-32B-Instruct"`, which is performant and available for fast inference, but you can select any compatible model from the Hub.

Running an agent is quite straightforward:

## 使用`smolagents`为派对选择播放列表

一场成功派对的重要组成部分是音乐。阿尔弗雷德需要一些帮助来挑选播放列表。幸运的是，`smolagents`能帮我们解决这个问题！我们可以构建一个能够搜索网络的智能体。为了让该智能体能够使用这个工具，我们在创建智能体时将其包含在工具列表中。

在模型方面，我们将依赖`InferenceClientModel`，它提供了对Hugging Face的[推理API](https://huggingface.co/docs/api-inference/index)的访问。默认模型是`"Qwen/Qwen2.5-Coder-32B-Instruct"`，该模型性能良好，可用于快速推理，但你也可以从模型库中选择任何兼容的模型。

运行智能体非常简单：

In [16]:
import os
from tavily import TavilyClient
from openai import OpenAI  # 改用OpenAI兼容接口调用Ollama（更稳定）
# 导入smolagents核心依赖（补充ChatMessage，适配新版接口）
from smolagents import CodeAgent, Tool, ChatMessage

# ===================== 1. 可配置参数（集中管理，便于修改） =====================
# Tavily API密钥
TAVILY_API_KEY = "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"
# 代理配置（无需代理则注释）
# HTTPS_PROXY = "http://127.0.0.1:7890"
# Ollama配置（本地/内网地址）
OLLAMA_BASE_URL = "http://127.0.0.1:11434/v1"
OLLAMA_MODEL_NAME = "qwen3:8b"  # 可替换为qwen3:8b/deepseek-r1:8b等

# 设置环境变量
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
# os.environ["HTTPS_PROXY"] = HTTPS_PROXY

# ===================== 2. 适配smolagents的Ollama模型类（核心修复） =====================
class OllamaModel:
    def __init__(self, model_name: str = OLLAMA_MODEL_NAME, base_url: str = OLLAMA_BASE_URL):
        """
        适配smolagents的Ollama模型封装（兼容新版smolagents的ChatMessage接口）
        :param model_name: 本地Ollama部署的模型名
        :param base_url: Ollama的OpenAI兼容接口地址
        """
        self.client = OpenAI(
            base_url=base_url,
            api_key="ollama"  # Ollama无需真实密钥，仅占位
        )
        self.model_name = model_name

    def generate(self, input_messages, stop_sequences=None, **additional_args) -> ChatMessage:
        """
        核心：适配smolagents的generate接口（输入是ChatMessage列表，输出也是ChatMessage）
        修复原简化版“单个prompt”的兼容问题
        """
        # 转换smolagents的ChatMessage为Ollama兼容格式
        openai_messages = []
        for msg in input_messages:
            openai_messages.append({
                "role": msg.role,
                "content": msg.content
            })
        
        # 调用本地Ollama模型（增加异常捕获）
        try:
            completion = self.client.chat.completions.create(
                model=self.model_name,
                messages=openai_messages,
                temperature=0.7,  # 控制生成随机性
                stream=False,
                **additional_args
            )
            # 返回smolagents要求的ChatMessage对象
            return ChatMessage(
                role="assistant",
                content=completion.choices[0].message.content.strip()
            )
        except Exception as e:
            return ChatMessage(
                role="assistant",
                content=f"模型调用失败：{str(e)}"
            )

# ===================== 3. 规范的Tavily搜索工具（增加错误处理） =====================
class CustomTavilySearchTool(Tool):
    def __init__(self):
        # 严格遵循smolagents的Tool规范（name/description/inputs必须定义）
        self.name = "web_search"
        self.description = "Searches the web for up-to-date music recommendations (e.g., 2025 Wayne mansion party music, villain themed party playlists)."
        self.inputs = {
            "query": {
                "type": "string",
                "description": "Music search query (e.g., 'best music for Wayne's mansion villain party 2025', '2025 popular party songs')."
            }
        }
        self.output_type = "string"
        super().__init__()
        self.client = TavilyClient(api_key=TAVILY_API_KEY)

    def forward(self, query: str) -> str:
        """执行搜索并返回格式化结果（增加异常捕获，避免程序崩溃）"""
        try:
            response = self.client.search(query=query, search_depth="basic")
            # 格式化结果（更易读，限制长度）
            results = "\n".join([f"- {r['title']}: {r['content'][:120]}..." for r in response["results"]])
            return results if results else "No music recommendations found in web search."
        except Exception as e:
            return f"Web search failed: {str(e)}"

# ===================== 4. 主函数（工程化封装，便于运行/调试） =====================
if __name__ == "__main__":
    # 1. 初始化模型和工具
    local_model = OllamaModel(model_name=OLLAMA_MODEL_NAME, base_url=OLLAMA_BASE_URL)
    search_tool = CustomTavilySearchTool()

    # 2. 初始化CodeAgent（增加日志级别、最大步骤限制）
    agent = CodeAgent(
        tools=[search_tool],
        model=local_model,
        max_steps=8,          # 限制最大执行步骤，防止死循环
        verbosity_level=2     # 详细日志，便于调试工具调用过程
    )

    # 3. 运行任务（Wayne庄园派对音乐推荐）
    task_prompt = "Search for the best music recommendations for a party at the Wayne's mansion (villain masquerade theme)."
    print(f"=== Starting Task: {task_prompt} ===")
    result = agent.run(task_prompt)

    # 4. 输出结果
    print("\n=== Final Music Recommendations ===")
    print(result)

=== Starting Task: Search for the best music recommendations for a party at the Wayne's mansion (villain masquerade theme). ===


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the best music recommendations for a party at the Wayne's mansion (villain masquerade theme).        │
│                                                                                                                 │
╰─ OllamaModel -  ────────────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
模型调用失败：Connection error.                                                                                    

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
Here is your code snippet:
模型调用失败：Connection error.</code>
Make sure to include code with the correct pattern, for instance:
Thoughts: Your thoughts
<code>
# Your python code here
</code>
Make sure to provide correct code blobs.

...

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
Here is your code snippet:
模型调用失败：Connection error.</code>
Make sure to include code with the correct pattern, for instance:
Thoughts: Your thoughts
<code>
# Your python code here
</code>
Make sure to provide correct code blobs.

[Step 8: Duration 7.53 seconds]

Reached max steps.

[Step 9: Duration 7.29 seconds]


=== Final Music Recommendations ===
模型调用失败：Connection error.


In [11]:
import os
from tavily import TavilyClient
from smolagents import CodeAgent, InferenceClientModel
from smolagents.tools import Tool

# 配置密钥与代理

os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["TAVILY_API_KEY"] = "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"

class CustomTavilySearchTool(Tool):
    def __init__(self):
        self.name = "web_search"
        self.description = "Searches the web for up-to-date information (e.g., trends, recommendations)."
        self.inputs = {
            "query": {
                "type": "string",
                "description": "The search query string (e.g., '2025 popular music')"
            }
        }
        self.output_type = "string"
        super().__init__()
        self.client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

    # 修正：执行方法命名为forward，参数明确为query（与inputs匹配）
    def forward(self, query: str) -> str:
        response = self.client.search(query=query, search_depth="basic")
        results = "\n".join([f"- {r['title']}: {r['content'][:100]}..." for r in response["results"]])
        return results if results else "No results found."

# 初始化并运行
search_tool = CustomTavilySearchTool()
agent = CodeAgent(tools=[search_tool], model=InferenceClientModel())
agent.run("Search for the best music recommendations for a party at the Wayne's mansion.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the best music recommendations for a party at the Wayne's mansion.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
402 Client Error: Payment Required for url: https://router.huggingface.co/novita/v3/openai/chat/completions 
(Request ID: Root=1-6982b04e-5f94145525ff4ca81cf421aa;90e73160-fb5f-4389-82d6-280bec752877)

Credit balance is depleted: purchase pre-paid credits to continue using Inference Providers with your account. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 1: Duration 0.64 seconds]

AgentGenerationError: Error in generating model output:
402 Client Error: Payment Required for url: https://router.huggingface.co/novita/v3/openai/chat/completions (Request ID: Root=1-6982b04e-5f94145525ff4ca81cf421aa;90e73160-fb5f-4389-82d6-280bec752877)

Credit balance is depleted: purchase pre-paid credits to continue using Inference Providers with your account. Alternatively, subscribe to PRO to get 20x more included usage.

In [9]:
from smolagents import CodeAgent, WebSearchTool, InferenceClientModel

agent = CodeAgent(tools=[WebSearchTool()], model=InferenceClientModel())

# agent.run("2025 Popular Music Recommendations.")
agent.run("2025 popular music")
# agent.run("Search for the best music recommendations for a party at the Wayne's mansion.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 2025 popular music                                                                                              │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  results = web_search(query="2025 popular music predictions")                                                     
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'results = web_search(query="2025 popular music predictions")' due to: Exception: No 
results found! Try a less restrictive/shorter query.

[Step 1: Duration 16.16 seconds| Input tokens: 2,084 | Output tokens: 1,729]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("No reliable information available for 2025 popular music as it is still in the future and current  
  search attempts are failing due to network issues.")                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: No reliable information available for 2025 popular music as it is still in the future and current 
search attempts are failing due to network issues.

[Step 3: Duration 24.42 seconds| Input tokens: 6,966 | Output tokens: 7,829]

'No reliable information available for 2025 popular music as it is still in the future and current search attempts are failing due to network issues.'

When you run this example, the output will **display a trace of the workflow steps being executed**. It will also print the corresponding Python code with the message:

```python
 ─ Executing parsed code: ────────────────────────────────────────────────────────────────────────────────────────
  results = web_search(query="best music for a Batman party")                                                      
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────
```

After a few steps, you'll see the generated playlist that Alfred can use for the party! 🎵


例如，输出将**显示正在执行的工作流步骤的跟踪信息**。它还会打印相应的Python代码，并附带以下信息：

```python
 ─ 正在执行解析后的代码： ────────────────────────────────────────────────────────────────────────────────────────
  results = web_search(query="适合蝙蝠侠主题派对的最佳音乐")                                                      
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────
```

经过几个步骤后，你将会看到生成的播放列表，阿尔弗雷德可以用它来准备派对！🎵

## Using a Custom Tool to Prepare the Menu

Now that we have selected a playlist, we need to organize the menu for the guests. Again, Alfred can take advantage of `smolagents` to do so. Here, we use the `@tool` decorator to define a custom function that acts as a tool. We'll cover tool creation in more detail later, so for now, we can simply run the code.

As you can see in the example below, we will create a tool using `@tool` decorator and include it in the `tools` list.

## 使用自定义工具准备菜单

既然我们已经选定了播放列表，就需要为客人安排菜单了。阿尔弗雷德同样可以利用`smolagents`来完成这项工作。在这里，我们使用`@tool`装饰器来定义一个用作工具的自定义函数。我们稍后会更详细地介绍工具的创建，所以现在只需运行代码即可。

正如你在下面的示例中所看到的，我们将使用`@tool`装饰器创建一个工具，并将其包含在`tools`列表中。

In [16]:
from smolagents import CodeAgent, tool

@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion (str): The type of occasion for the party. Allowed values are:
                        - "casual": Menu for casual party.
                        - "formal": Menu for formal party.
                        - "superhero": Menu for superhero party.
                        - "custom": Custom menu.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

agent = CodeAgent(tools=[suggest_menu], model=InferenceClientModel())

agent.run("Prepare a formal menu for the party.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Prepare a formal menu for the party.                                                                            │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  menu = suggest_menu(occasion="formal")                                                                           
  final_answer(menu)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 3-course dinner with wine and dessert.

[Step 1: Duration 5.98 seconds| Input tokens: 2,112 | Output tokens: 396]

'3-course dinner with wine and dessert.'

The agent will run for a few steps until finding the answer.

The menu is ready! 🥗

智能体将运行几步，直到找到答案。

菜单准备好了！🥗

## Using Python Imports Inside the Agent

We have the playlist and menu ready, but we need to check one more crucial detail: preparation time!

Alfred needs to calculate when everything would be ready if he started preparing now, in case they need assistance from other superheroes.

`smolagents` specializes in agents that write and execute Python code snippets, offering sandboxed execution for security. It supports both open-source and proprietary language models, making it adaptable to various development environments.

**Code execution has strict security measures** - imports outside a predefined safe list are blocked by default. However, you can authorize additional imports by passing them as strings in `additional_authorized_imports`.
For more details on secure code execution, see the official [guide](https://huggingface.co/docs/smolagents/tutorials/secure_code_execution).

When creating the agent, we will use `additional_authorized_imports` to allow for importing the `datetime` module.

## 在智能体内部使用Python导入

我们已经准备好播放列表和菜单，但还需要检查一个更关键的细节：准备时间！

阿尔弗雷德需要计算如果现在开始准备，所有东西什么时候能准备好，以防他们需要其他超级英雄的帮助。

`smolagents` 专注于那些编写和执行Python代码片段的智能体，提供沙盒执行以确保安全性。它同时支持开源和专有语言模型，使其能够适应各种开发环境。

**代码执行有严格的安全措施**——默认情况下，预定义安全列表之外的导入会被阻止。不过，你可以通过在 `additional_authorized_imports` 中以字符串形式传入额外的导入内容来授权它们。
有关安全代码执行的更多详细信息，请参阅官方[指南](https://huggingface.co/docs/smolagents/tutorials/secure_code_execution)。

在创建智能体时，我们将使用 `additional_authorized_imports` 来允许导入 `datetime` 模块。

In [18]:
from smolagents import CodeAgent, InferenceClientModel
import numpy as np
import time
import datetime

agent = CodeAgent(tools=[], model=InferenceClientModel(), additional_authorized_imports=['datetime'])

agent.run(
    """
    Alfred needs to prepare for the party. Here are the tasks:
    1. Prepare the drinks - 30 minutes
    2. Decorate the mansion - 60 minutes
    3. Set up the menu - 45 minutes
    3. Prepare the music and playlist - 45 minutes

    If we start right now, at what time will the party be ready?
    """
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Alfred needs to prepare for the party. Here are the tasks:                                                      │
│     1. Prepare the drinks - 30 minutes                                                                          │
│     2. Decorate the mansion - 60 minutes                                                                        │
│     3. Set up the menu - 45 minutes                                                                             │
│     3. Prepare the music and playlist - 45 minutes                                                              │
│                                                                                                                 │
│     If we start right now, at what time will the party be ready?                                                │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  total_minutes = 30 + 60 + 45 + 45                                                                                
  hours = total_minutes // 60                                                                                      
  final_answer(f"{hours} hours from now")                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 3 hours from now

[Step 1: Duration 26.66 seconds| Input tokens: 2,111 | Output tokens: 3,820]

'3 hours from now'

These examples are just the beginning of what you can do with code agents, and we're already starting to see their utility for preparing the party.
You can learn more about how to build code agents in the [smolagents documentation](https://huggingface.co/docs/smolagents).

`smolagents` specializes in agents that write and execute Python code snippets, offering sandboxed execution for security. It supports both open-source and proprietary language models, making it adaptable to various development environments.

这些例子仅仅是你使用代码代理所能做之事的开端，而且我们已经开始看到它们在筹备派对方面的实用性。

你可以在[smolagents文档](https://huggingface.co/docs/smolagents)中了解更多关于如何构建代码代理的信息。

`smolagents`专门研究那些编写和执行Python代码片段的代理，为安全起见提供沙盒执行环境。它既支持开源语言模型，也支持专有语言模型，从而能够适应各种开发环境。

## Sharing Our Custom Party Preparator Agent to the Hub

Wouldn't it be **amazing to share our very own Alfred agent with the community**? By doing so, anyone can easily download and use the agent directly from the Hub, bringing the ultimate party planner of Gotham to their fingertips! Let's make it happen! 🎉

The `smolagents` library makes this possible by allowing you to share a complete agent with the community and download others for immediate use. It's as simple as the following:

## 向中心分享我们的定制派对筹备代理

能把我们自己的阿尔弗雷德代理分享给社区，这难道不**很棒**吗？通过这样做，任何人都能轻松地从中心下载并直接使用这个代理，让哥谭市的终极派对策划者触手可及！让我们实现它吧！🎉

`smolagents`库让这一切成为可能，它允许你向社区分享一个完整的代理，也能下载其他代理立即使用。过程就像下面这样简单：

In [23]:
import os
from tavily import TavilyClient
from smolagents.tools import Tool

# 配置代理（保持你的有效代理）
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
# 填入复制的Tavily API密钥
os.environ["TAVILY_API_KEY"] = "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"

class CustomTavilySearchTool(Tool):
    def __init__(self):
        self.name = "web_search"
        self.description = "Searches the web for up-to-date information (e.g., trends, recommendations)."
        self.inputs = {
            "query": {
                "type": "string",
                "description": "The search query string (e.g., '2025 popular music')"
            }
        }
        self.output_type = "string"
        super().__init__()
        self.client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

    # 修正：执行方法命名为forward，参数明确为query（与inputs匹配）
    def forward(self, query: str) -> str:
        response = self.client.search(query=query, search_depth="basic")
        results = "\n".join([f"- {r['title']}: {r['content'][:100]}..." for r in response["results"]])
        return results if results else "No results found."

# 初始化并运行
# search_tool = CustomTavilySearchTool()
# agent = CodeAgent(tools=[search_tool], model=InferenceClientModel())
# agent.run("Search for the best music recommendations for a party at the Wayne's mansion.")

In [3]:
# 导入核心依赖
import os
from tavily import TavilyClient
from openai import OpenAI

# 导入smolagents相关模块（仅保留存在的工具/类）
from smolagents import CodeAgent, Tool, tool, ChatMessage
from smolagents import FinalAnswerTool

# ===================== 1. 配置关键参数（需替换为你的实际信息） =====================
# Tavily API密钥（从Tavily官网获取）
TAVILY_API_KEY = "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"
# 代理配置（无需代理则注释/删除）
HTTPS_PROXY = "http://127.0.0.1:7890"
# Ollama服务地址（确保Ollama已启动，地址正确）
OLLAMA_BASE_URL = "http://172.19.120.145:11434/v1"
# Ollama模型名称
OLLAMA_MODEL_NAME = "qwen3:8b"
# OLLAMA_MODEL_NAME = "deepseek-r1:8b"

# 设置环境变量
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["HTTPS_PROXY"] = HTTPS_PROXY

# ===================== 2. 自定义Ollama模型类（适配smolagents） =====================
class OllamaModel:
    def __init__(self, model_name=OLLAMA_MODEL_NAME, base_url=OLLAMA_BASE_URL):
        # 初始化OpenAI客户端（对接Ollama的OpenAI兼容接口）
        self.client = OpenAI(
            base_url=base_url,
            api_key="ollama"  # Ollama无需真实密钥，仅满足格式要求
        )
        self.model_name = model_name

    def generate(self, input_messages, stop_sequences=None, **additional_args) -> ChatMessage:
        """
        适配smolagents的模型调用接口，返回ChatMessage对象
        :param input_messages: smolagents传入的消息列表（ChatMessage类型）
        :param stop_sequences: 停止序列（smolagents自动传递，此处忽略）
        :param additional_args: 额外参数（透传给Ollama）
        :return: ChatMessage对象（符合smolagents要求）
        """
        # 转换smolagents的ChatMessage为Ollama兼容的格式
        openai_messages = []
        for msg in input_messages:
            openai_messages.append({
                "role": msg.role,
                "content": msg.content
            })
        
        # 调用Ollama生成响应
        completion = self.client.chat.completions.create(
            model=self.model_name,
            messages=openai_messages,
            temperature=0.7,  # 控制生成的随机性
            **additional_args
        )
        
        # 构造并返回ChatMessage对象（核心修复点）
        return ChatMessage(
            role="assistant",
            content=completion.choices[0].message.content
        )

# ===================== 3. 自定义Tavily搜索工具（符合smolagents规范） =====================
class CustomTavilySearchTool(Tool):
    def __init__(self):
        # Tool类强制属性（name/description/inputs/output_type）
        self.name = "web_search"
        self.description = "Searches the web for up-to-date information (e.g., villain masquerade party playlists, Batman villain themed songs)."
        self.inputs = {
            "query": {
                "type": "string",
                "description": "Search query (e.g., 'best villain masquerade party playlist 2025', 'Batman villain themed songs')."
            }
        }
        self.output_type = "string"
        super().__init__()
        # 初始化Tavily客户端
        self.client = TavilyClient(api_key=TAVILY_API_KEY)

    def forward(self, query: str) -> str:
        """执行搜索并返回格式化结果"""
        try:
            response = self.client.search(query=query, search_depth="basic")
            # 格式化搜索结果（仅保留标题+前150字符内容，避免过长）
            results = "\n".join([f"- {r['title']}: {r['content'][:150]}..." for r in response["results"]])
            return results if results else "No search results found."
        except Exception as e:
            return f"Search failed: {str(e)}"

# ===================== 4. 原有自定义工具（保留并确保规范） =====================
@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion: The type of occasion for the party (e.g., casual, formal, superhero).
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

@tool
def catering_service_tool(query: str) -> str:
    """
    Returns the highest-rated catering service in Gotham City.
    Args:
        query: Search term for catering services (e.g., 'Gotham catering', 'Wayne Manor catering').
    """
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }
    best_service = max(services, key=services.get)
    return f"Highest-rated catering service: {best_service} (Rating: {services[best_service]})"

class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = "Suggests creative superhero-themed party ideas based on a category (returns unique theme ideas)."
    inputs = {
        "category": {
            "type": "string",
            "description": "Party category (e.g., 'classic heroes', 'villain masquerade', 'futuristic gotham').",
        }
    }
    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests dress as DC heroes with 'Kryptonite Punch' cocktails.",
            "villain masquerade": "Gotham Rogues' Ball: Mysterious masquerade for Batman villains (e.g., Joker, Catwoman).",
            "futuristic gotham": "Neo-Gotham Night: Cyberpunk party inspired by Batman Beyond (neon decor + futuristic gadgets)."
        }
        return themes.get(category.lower(), 
                          "Theme not found. Try: 'classic heroes', 'villain masquerade', 'futuristic gotham'.")

# ===================== 5. 初始化Agent并运行任务 =====================
if __name__ == "__main__":
    # 初始化本地Ollama模型
    local_model = OllamaModel(model_name=OLLAMA_MODEL_NAME, base_url=OLLAMA_BASE_URL)
    
    # 初始化CodeAgent（仅保留存在的工具）
    agent = CodeAgent(
        tools=[
            CustomTavilySearchTool(),    # 网页搜索工具
            suggest_menu,                # 菜单推荐工具
            catering_service_tool,       # 餐饮服务工具
            SuperheroPartyThemeTool()    # 派对主题工具
        ],
        model=local_model,       # 使用本地Ollama模型（替代付费的InferenceClientModel）
        max_steps=10,            # 最大执行步骤
        verbosity_level=2        # 详细日志级别（便于调试）
    )

    # 运行任务：生成Wayne庄园反派化妆舞会的最佳歌单
    task_prompt = "Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme"
    print(f"=== Running Task: {task_prompt} ===")
    result = agent.run(task_prompt)
    
    # 输出最终结果
    print("\n=== Final Result ===")
    print(result)

=== Running Task: Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme ===


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme        │
│                                                                                                                 │
╰─ OllamaModel -  ────────────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
<think>                                                                                                            
Okay, let's tackle this task. The user wants the best playlist for a party at Wayne's mansion with a 'villain      
masquerade' theme. First, I need to figure out the right approach.                                                 
                                                                                                                   
Hmm, the tools available include web_search, suggest_menu, catering_service_tool, superhero_party_theme_generator, 
and final_answer. The user mentioned a villain masquerade theme, so maybe the superhero_party_theme_generator could
help with themes, but the main need is a playlist. Wait, the tools don't have a direct music playlist generator.   
But there's a web_search tool that can look up playlists. Also, the superhero_party_theme_generator might provide  
some theme ideas that could influence the playlist.                                                                
                                                                                                                   
Wait, the user specifically asked for a playlist. Since there's no direct tool for playlists, maybe I can use      
web_search to find "best villain masquerade party playlist 2025" or similar. Alternatively, the                    
superhero_party_theme_generator with category 'villain masquerade' might suggest themes that can be translated into
music ideas. But the user wants a playlist, so perhaps the best approach is to use web_search with a query tailored
to that.                                                                                                           
                                                                                                                   
Let me check the web_search tool's description. It says it can search for up-to-date info, like villain masquerade 
party playlists. So using

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  playlist = web_search(query="best villain masquerade party playlist 2025")                                       
  print(playlist)                                                                                                  
                                                                                                                   
  final_answer(playlist)                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
- Songs That Make You Feel Like a Legendary Villain ⚔️ Playlist 2025: Songs That Make You Feel Like a Legendary 
Villain 😈⚔️ Playlist 2025 🔥
MrMusicMix
12400 subscribers
200 likes
13820 views
8 May 2024
Songs That Make Yo...
- EPIC MENACING VILLAIN MUSIC MIX 2025 - YouTube: EPIC MENACING VILLAIN MUSIC MIX 2025 🔥
Content Chimp
102000 subscribers
1322 likes
94356 views
2 Oct 2022
Embrace your dark side with our playlist of ...
- VILLAIN ERA 2025 - playlist by Jordan Fiction - Spotify: VILLAIN ERA 2025 ; KILLING TIME - Exhumed · Jordan 
Fiction ; DIE. Johnny Goth ; HATE PARTY · bodyimage, DeathbyRomy ; Pink Rover · Scene Queen ; Every...
- Villain Party Playlist to Boost Your Confidence❤️‍  | HipHop, Dark ...: Villain Party Playlist to Boost Your 
Confidence❤️‍ | HipHop, Dark ... 2025 Hip Hop- Twerk Party Mix- GloRilla, Latto, Nicki Minaj...
- villain era 2025 - playlist by DistroCat | Spotify: villain era 2025. villain vibes - bad bitch vibes - in my 
villain era - villaincore - new year new villain - villainess energy - villain mode - 2024 v...

Final answer: - Songs That Make You Feel Like a Legendary Villain ⚔️ Playlist 2025: Songs That Make You Feel Like a
Legendary Villain 😈⚔️ Playlist 2025 🔥
MrMusicMix
12400 subscribers
200 likes
13820 views
8 May 2024
Songs That Make Yo...
- EPIC MENACING VILLAIN MUSIC MIX 2025 - YouTube: EPIC MENACING VILLAIN MUSIC MIX 2025 🔥
Content Chimp
102000 subscribers
1322 likes
94356 views
2 Oct 2022
Embrace your dark side with our playlist of ...
- VILLAIN ERA 2025 - playlist by Jordan Fiction - Spotify: VILLAIN ERA 2025 ; KILLING TIME - Exhumed · Jordan 
Fiction ; DIE. Johnny Goth ; HATE PARTY · bodyimage, DeathbyRomy ; Pink Rover · Scene Queen ; Every...
- Villain Party Playlist to Boost Your Confidence❤️‍  | HipHop, Dark ...: Villain Party Playlist to Boost Your 
Confidence❤️‍ | HipHop, Dark ... 2025 Hip Hop- Twerk Party Mix- GloRilla, Latto, Nicki Minaj...
- villain era 2025 - playlist by DistroCat | Spotify: villain era 2025. villain vibes - bad bitch vibes - in my 
villain era - villaincore - new year new villain - villainess energy - villain mode - 2024 v...

[Step 1: Duration 19.12 seconds]


=== Final Result ===
- Songs That Make You Feel Like a Legendary Villain ⚔️ Playlist 2025: Songs That Make You Feel Like a Legendary Villain 😈⚔️ Playlist 2025 🔥
MrMusicMix
12400 subscribers
200 likes
13820 views
8 May 2024
Songs That Make Yo...
- EPIC MENACING VILLAIN MUSIC MIX 2025 - YouTube: EPIC MENACING VILLAIN MUSIC MIX 2025 🔥
Content Chimp
102000 subscribers
1322 likes
94356 views
2 Oct 2022
Embrace your dark side with our playlist of ...
- VILLAIN ERA 2025 - playlist by Jordan Fiction - Spotify: VILLAIN ERA 2025 ; KILLING TIME - Exhumed · Jordan Fiction ; DIE. Johnny Goth ; HATE PARTY · bodyimage, DeathbyRomy ; Pink Rover · Scene Queen ; Every...
- Villain Party Playlist to Boost Your Confidence❤️‍  | HipHop, Dark ...: Villain Party Playlist to Boost Your Confidence❤️‍ | HipHop, Dark ... 2025 Hip Hop- Twerk Party Mix- GloRilla, Latto, Nicki Minaj...
- villain era 2025 - playlist by DistroCat | Spotify: villain era 2025. villain vibes - bad bitch vibes - in my villain era - vill

In [24]:
from smolagents import CodeAgent, InferenceClientModel, VisitWebpageTool, FinalAnswerTool, Tool, tool

@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion: The type of occasion for the party.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.

    Args:
        query: A search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    # Find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)

    return best_service

class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""

    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic gotham').",
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }

        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic gotham'.")


# Alfred, the butler, preparing the menu for the party
agent = CodeAgent(
    tools=[
        CustomTavilySearchTool(),
        VisitWebpageTool(),
        suggest_menu,
        catering_service_tool,
        SuperheroPartyThemeTool()
        ],
    model=InferenceClientModel(),
    max_steps=10,
    verbosity_level=2
)

agent.run("Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme        │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
402 Client Error: Payment Required for url: https://router.huggingface.co/novita/v3/openai/chat/completions 
(Request ID: Root=1-6982a24b-66ccbc467e2dbbd61758bf02;4eaafc52-f374-4aa6-a4b4-9a51ee365cac)

Credit balance is depleted: purchase pre-paid credits to continue using Inference Providers with your account. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 1: Duration 0.78 seconds]

AgentGenerationError: Error in generating model output:
402 Client Error: Payment Required for url: https://router.huggingface.co/novita/v3/openai/chat/completions (Request ID: Root=1-6982a24b-66ccbc467e2dbbd61758bf02;4eaafc52-f374-4aa6-a4b4-9a51ee365cac)

Credit balance is depleted: purchase pre-paid credits to continue using Inference Providers with your account. Alternatively, subscribe to PRO to get 20x more included usage.

In [8]:
# agent.push_to_hub('sergiopaniego/AlfredAgent')
# 修改后（关键：添加create_pr=True）
agent.push_to_hub(
    repo_id="jiaendu/AlfredAgent"
    # create_pr=True  # 强制通过创建PR提交，解决403拦截
    # token="你的Hugging Face写入权限Token"  # 替换为你在Access Tokens中生成的有效Token
)

README.md:   0%|          | 0.00/230 [00:00<?, ?B/s]

d:\Users\jiaendu\Anaconda3\envs\py311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jiaendu\.cache\huggingface\hub\spaces--jiaendu--AlfredAgent. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


ValueError: Tool validation failed for CustomTavilySearchTool:
- __init__: Name 'TavilyClient' is undefined.
- __init__: Name 'TavilyClient' is undefined.
- __init__: Name 'TAVILY_API_KEY' is undefined.

To download the agent again, use the code below:

In [ ]:
agent = CodeAgent(tools=[], model=InferenceClientModel())
alfred_agent = agent.from_hub('sergiopaniego/AlfredAgent', trust_remote_code=True)

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

.gitattributes:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/258 [00:00<?, ?B/s]

requirements.txt:   0%|          | 0.00/50.0 [00:00<?, ?B/s]

app.py:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

...

prompts.yaml:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

tools%2Fsuggest_menu.py:   0%|          | 0.00/822 [00:00<?, ?B/s]

(…)ols%2Fsuperhero_party_theme_generator.py:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tools%2Fweb_search.py:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

tools%2Fvisit_webpage.py:   0%|          | 0.00/1.82k [00:00<?, ?B/s]

In [ ]:
alfred_agent.run("Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme        │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: First, I will generate a creative superhero-themed party idea for a 'villain masquerade' theme using the  
`superhero_party_theme_generator` tool. Then, I will use the `web_search` tool to find the best playlist for a     
villain masquerade party theme at Wayne's mansion.                                                                 
                                                                                                                   
Code:                                                                                                              
```py                                                                                                              
party_theme = superhero_party_theme_generator(category="villain masquerade")                                       
print(f"Suggested party theme: {party_theme}")                                                                     
```<end_code>                                                                                                      

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  party_theme = superhero_party_theme_generator(category="villain masquerade")                                     
  print(f"Suggested party theme: {party_theme}")                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Suggested party theme: Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: It seems there is a persistent issue with the `visit_webpage` tool in this environment. Given that the URL
we identified is likely a valid Spotify playlist URL, we can proceed by directly suggesting that URL as the best   
playlist for the "villain masquerade" theme at Wayne's mansion.                                                    
                                                                                                                   
Code:                                                                                                              
```py                                                                                                              
final_answer("https://open.spotify.com/playlist/3cejFigsE9RrSdG4xUCmay")                                           
```<end_code>                                                                                                      

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("https://open.spotify.com/playlist/3cejFigsE9RrSdG4xUCmay")                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: https://open.spotify.com/playlist/3cejFigsE9RrSdG4xUCmay

[Step 4: Duration 7.82 seconds| Input tokens: 16,649 | Output tokens: 551]

'https://open.spotify.com/playlist/3cejFigsE9RrSdG4xUCmay'

What's also exciting is that shared agents are directly available as Hugging Face Spaces, allowing you to interact with them in real-time. You can explore other agents [here](https://huggingface.co/spaces/davidberenstein1957/smolagents-and-tools).

For example, the _AlfredAgent_ is available [here](https://huggingface.co/spaces/sergiopaniego/AlfredAgent).

### Inspecting Our Party Preparator Agent with OpenTelemetry and Langfuse 📡

Full trace can be found [here](https://cloud.langfuse.com/project/cm7bq0abj025rad078ak3luwi/traces/995fc019255528e4f48cf6770b0ce27b?timestamp=2025-02-19T10%3A28%3A36.929Z).

As Alfred fine-tunes the Party Preparator Agent, he's growing weary of debugging its runs. Agents, by nature, are unpredictable and difficult to inspect. But since he aims to build the ultimate Party Preparator Agent and deploy it in production, he needs robust traceability for future monitoring and analysis.  

Once again, `smolagents` comes to the rescue! It embraces the [OpenTelemetry](https://opentelemetry.io/) standard for instrumenting agent runs, allowing seamless inspection and logging. With the help of [Langfuse](https://langfuse.com/) and the `SmolagentsInstrumentor`, Alfred can easily track and analyze his agent’s behavior.  

Setting it up is straightforward!  

First, we need to install the necessary dependencies:  

In [ ]:
!pip install smolagents[telemetry] opentelemetry-sdk opentelemetry-exporter-otlp openinference-instrumentation-smolagents

Next, Alfred has already created an account on Langfuse and has his API keys ready. If you haven’t done so yet, you can sign up for Langfuse Cloud [here](https://cloud.langfuse.com/) or explore [alternatives](https://huggingface.co/docs/smolagents/tutorials/inspect_runs).  

Once you have your API keys, they need to be properly configured as follows:

In [ ]:
import os
import base64
from google.colab import userdata

LANGFUSE_PUBLIC_KEY=userdata.get("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY=userdata.get("LANGFUSE_SECRET_KEY")
LANGFUSE_AUTH=base64.b64encode(f"{LANGFUSE_PUBLIC_KEY}:{LANGFUSE_SECRET_KEY}".encode()).decode()

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://cloud.langfuse.com/api/public/otel" # EU data region
# os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://us.cloud.langfuse.com/api/public/otel" # US data region
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"

Finally, Alfred is ready to initialize the `SmolagentsInstrumentor` and start tracking his agent's performance.  

In [ ]:
from opentelemetry.sdk.trace import TracerProvider

from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

trace_provider = TracerProvider()
trace_provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter()))

SmolagentsInstrumentor().instrument(tracer_provider=trace_provider)

Alfred is now connected 🔌! The runs from `smolagents` are being logged in Langfuse, giving him full visibility into the agent's behavior. With this setup, he's ready to revisit previous runs and refine his Party Preparator Agent even further.  

In [ ]:
from smolagents import CodeAgent, InferenceClientModel

agent = CodeAgent(tools=[], model=InferenceClientModel())
alfred_agent = agent.from_hub('sergiopaniego/AlfredAgent', trust_remote_code=True)
alfred_agent.run("Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme")

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme        │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: First, I will generate a creative superhero-themed party idea for a 'villain masquerade' theme using the  
`superhero_party_theme_generator` tool. Then, I will use the `web_search` tool to find the best playlist for a     
villain masquerade party theme at Wayne's mansion.                                                                 
                                                                                                                   
Code:                                                                                                              
```py                                                                                                              
party_theme = superhero_party_theme_generator(category="villain masquerade")                                       
print(f"Suggested party theme: {party_theme}")                                                                     
```<end_code>                                                                                                      

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  party_theme = superhero_party_theme_generator(category="villain masquerade")                                     
  print(f"Suggested party theme: {party_theme}")                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: It seems there was an issue with visiting the Spotify URL. Since visiting the webpage directly didn't     
work, I'll provide the Spotify playlist URL directly as the final answer based on the search results.              
                                                                                                                   
Final Answer:                                                                                                      
                                                                                                                   
Code:                                                                                                              
```py                                                                                                              
final_answer("https://open.spotify.com/playlist/0gZMMHjuxMrrybQ7wTMTpw")                                           
```<end_code>                                                                                                      

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("https://open.spotify.com/playlist/0gZMMHjuxMrrybQ7wTMTpw")                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: https://open.spotify.com/playlist/0gZMMHjuxMrrybQ7wTMTpw

[Step 3: Duration 6.05 seconds| Input tokens: 12,474 | Output tokens: 410]

'https://open.spotify.com/playlist/0gZMMHjuxMrrybQ7wTMTpw'

Alfred can now access this logs [here](https://cloud.langfuse.com/project/cm7bq0abj025rad078ak3luwi/traces/995fc019255528e4f48cf6770b0ce27b?timestamp=2025-02-19T10%3A28%3A36.929Z) to review and analyze them.  

Meanwhile, the [suggested playlist](https://open.spotify.com/playlist/0gZMMHjuxMrrybQ7wTMTpw) sets the perfect vibe for the party preparations. Cool, right? 🎶
